In [ ]:
# Prometheus logs analysis

In [ ]:
import requests
import os

# 1. Define your list of nodes
# Add any other node names here (e.g. 'alice', 'bob', 'charlie')
RELAYS = ["ferdie", "george", "henry", "iris", "jack", "paul", "quinn", "rita", "sam", "tom"]
BLOCK_PRODUCERS = ["alice", "bob", "charlie", "dave", "eve", "kate", "leo", "mike", "nina", "oliver"]
NODES = RELAYS + BLOCK_PRODUCERS

# 2. Define the pattern
base_url = "http://{name}.node.sc.iog.io:9615/metrics"

print(f"Starting download for {len(NODES)} nodes...\n")

for node in NODES:
    # Construct the specific URL and Filename
    url = base_url.format(name=node)
    filename = f"prometheus_{node}.log"
    
    try:
        print(f"📡 Fetching {node}...", end=" ")
        
        # Perform the request (timeout ensures it doesn't hang forever)
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raises error if 404/500
        
        # Save to file
        with open(filename, "w", encoding="utf-8") as f:
            f.write(response.text)
            
        print(f"✅ Saved to {filename}")
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Failed: {e}")

print("\nAll done.")

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
# We set this to "." to scan the current folder where you just saved the files
LOG_FOLDER = "."  
FILE_EXTENSION = ".log" 

# Metric names to hunt for
METRIC_PROP = "substrate_proposer_block_proposal_time"
METRIC_IMPORT = "substrate_block_verification_and_import_time"

# ==========================================
# PARSER LOGIC
# ==========================================
def parse_prometheus_buckets(file_content, metric_name):
    """
    Extracts bucket counts for a specific metric from raw Prometheus text.
    """
    # Regex to capture: metric_bucket{... le="0.005"} 123
    pattern = re.compile(rf'{metric_name}_bucket\{{.*le="([\d\.\+Inf]+)".*\}} (\d+)')
    
    bucket_map = {}
    
    for line in file_content.splitlines():
        if metric_name + "_bucket" not in line:
            continue
            
        match = pattern.search(line)
        if match:
            le_str = match.group(1)
            # Handle standard Prometheus "+Inf"
            if "+Inf" in le_str:
                le_val = float('inf')
            else:
                try:
                    le_val = float(le_str)
                except ValueError:
                    continue
            
            count = int(float(match.group(2)))
            
            # Use max count found (cumulative counter logic)
            if le_val in bucket_map:
                bucket_map[le_val] = max(bucket_map[le_val], count)
            else:
                bucket_map[le_val] = count
                
    return bucket_map

def aggregate_logs(folder, extension):
    total_prop_buckets = {}
    total_import_buckets = {}
    files_found = 0
    
    print(f"Scanning '{folder}' for *{extension} files...")
    
    for filename in os.listdir(folder):
        if filename.endswith(extension) and filename.startswith("prometheus_"):
            files_found += 1
            path = os.path.join(folder, filename)
            
            try:
                with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    
                # Parse file
                prop_data = parse_prometheus_buckets(content, METRIC_PROP)
                import_data = parse_prometheus_buckets(content, METRIC_IMPORT)
                
                # Merge into totals
                for le, count in prop_data.items():
                    total_prop_buckets[le] = total_prop_buckets.get(le, 0) + count
                    
                for le, count in import_data.items():
                    total_import_buckets[le] = total_import_buckets.get(le, 0) + count
                    
            except Exception as e:
                print(f"Error reading {filename}: {e}")
                
    print(f"aggregated metrics from {files_found} log files.")
    return total_prop_buckets, total_import_buckets

def buckets_to_intervals(bucket_map):
    if not bucket_map:
        return [], []
    
    sorted_buckets = sorted(bucket_map.items(), key=lambda x: x[0])
    
    intervals = []
    counts = []
    prev_count = 0
    prev_le = 0.0
    
    for le_val, total_count in sorted_buckets:
        current_interval_count = total_count - prev_count
        if current_interval_count < 0: current_interval_count = 0 
        
        if le_val == float('inf'):
            label = f"> {prev_le}s"
        else:
            label = f"{prev_le}-{le_val}s"
            
        intervals.append(label)
        counts.append(current_interval_count)
        
        prev_count = total_count
        prev_le = le_val
        
    return intervals, counts

# ==========================================
# MAIN EXECUTION
# ==========================================

# 1. Aggregate Data
prop_buckets, import_buckets = aggregate_logs(LOG_FOLDER, FILE_EXTENSION)

# 2. Process into Intervals
p_intervals, p_counts = buckets_to_intervals(prop_buckets)
i_intervals, i_counts = buckets_to_intervals(import_buckets)

# 3. Create DataFrame
df_prop = pd.DataFrame({'Interval': p_intervals, 'Proposal_Count': p_counts})
df_import = pd.DataFrame({'Interval': i_intervals, 'Import_Count': i_counts})

if df_prop.empty and df_import.empty:
    print("⚠️ No Prometheus data found. Did the download script run successfully?")
else:
    # Merge on Interval
    df = pd.merge(df_prop, df_import, on='Interval', how='outer').fillna(0)

    # Sort helper
    def extract_upper(x):
        if ">" in x: return float('inf')
        try: return float(x.split('-')[1].replace('s',''))
        except: return 0

    df['sort'] = df['Interval'].apply(extract_upper)
    df = df.sort_values('sort').drop('sort', axis=1)

    # 4. Plot
    plt.figure(figsize=(14, 7))
    bar_width = 0.35
    index = np.arange(len(df))

    # Log scale is crucial for viewing outliers
    plt.yscale('log')

    # Bars
    p_bars = plt.bar(index, df['Proposal_Count'], bar_width, label='Block Creation (CPU)', color='skyblue', edgecolor='black')
    i_bars = plt.bar(index + bar_width, df['Import_Count'], bar_width, label='Block Import (Disk/Net)', color='salmon', edgecolor='black')

    # Styling
    plt.xlabel('Duration Buckets', fontsize=12)
    plt.ylabel('Count of Blocks (Log Scale)', fontsize=12)
    plt.title('Block Processing Latency (All Nodes Aggregated)', fontsize=16)
    plt.xticks(index + bar_width / 2, df['Interval'], rotation=45)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    # Annotate significant outliers (only show numbers for buckets > 0.5s)
    def annotate_bars(bars, data_col):
        for i, rect in enumerate(bars):
            height = rect.get_height()
            label = df.iloc[i]['Interval']
            
            # Show number if count > 0 AND it's a slow bucket
            is_slow = any(x in label for x in ["0.5-", "1.0-", "2.5-", "5.0-", ">"])
            
            if height > 0 and is_slow:
                plt.text(rect.get_x() + rect.get_width()/2., height * 1.1,
                        f'{int(height)}',
                        ha='center', va='bottom', fontsize=10, fontweight='bold')

    annotate_bars(p_bars, 'Proposal_Count')
    annotate_bars(i_bars, 'Import_Count')

    plt.tight_layout()
    plt.show()

    # Optional: Print the raw table for the tail
    print("\n--- Outlier Detail (Slowest Blocks) ---")
    print(df[df['Interval'].str.contains("0.5|1.0|2.5|5.0|>")])

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
# Set to "." to scan the current folder, or provide a specific path
LOG_FOLDER = "."  
FILE_EXTENSION = ".log" 

# List of metrics to analyze (Metric Name, Display Title)
METRICS_CONFIG = [
    ("ledger_txs_processing_time", "Total Processing Time"),
    ("ledger_txs_validating_time", "Validation Time"),
    ("storage_flush_time", "Storage Flush Time")  # Changed from storage_fetch_time
]

# ==========================================
# PARSER LOGIC
# ==========================================
def parse_prometheus_buckets(file_content, metric_name):
    """
    Extracts bucket counts for a specific metric from raw Prometheus text.
    Returns a dict: {le_value: count}
    """
    # Regex to capture: metric_bucket{... le="0.005"} 123
    pattern = re.compile(rf'{metric_name}_bucket\{{.*le="([\d\.\+Inf]+)".*\}} (\d+)')
    
    bucket_map = {}
    
    for line in file_content.splitlines():
        if metric_name + "_bucket" not in line:
            continue
            
        match = pattern.search(line)
        if match:
            le_str = match.group(1)
            # Handle standard Prometheus "+Inf"
            if "+Inf" in le_str:
                le_val = float('inf')
            else:
                try:
                    le_val = float(le_str)
                except ValueError:
                    continue
            
            count = int(float(match.group(2)))
            
            # Use max count found in this specific file (cumulative counter logic)
            if le_val in bucket_map:
                bucket_map[le_val] = max(bucket_map[le_val], count)
            else:
                bucket_map[le_val] = count
                
    return bucket_map

def aggregate_logs(folder, extension, metrics_list):
    """
    Scans folder for files and aggregates buckets for all requested metrics.
    Returns a dict: {metric_name: {le_value: total_count}}
    """
    aggregated_data = {m_name: {} for m_name, _ in metrics_list}
    files_found = 0
    
    print(f"Scanning '{folder}' for *{extension} files...")
    
    for filename in os.listdir(folder):
        if filename.endswith(extension) and "prometheus" in filename:
            files_found += 1
            path = os.path.join(folder, filename)
            
            try:
                with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                
                for metric_name, _ in metrics_list:
                    file_buckets = parse_prometheus_buckets(content, metric_name)
                    for le, count in file_buckets.items():
                        current_total = aggregated_data[metric_name].get(le, 0)
                        aggregated_data[metric_name][le] = current_total + count
                        
            except Exception as e:
                print(f"Error reading {filename}: {e}")
                
    print(f"Aggregated metrics from {files_found} log files.")
    return aggregated_data

def buckets_to_intervals(bucket_map):
    """
    Converts cumulative buckets (le) into non-cumulative interval counts.
    """
    if not bucket_map:
        return [], []
    
    sorted_buckets = sorted(bucket_map.items(), key=lambda x: x[0])
    
    intervals = []
    counts = []
    prev_count = 0
    prev_le = 0.0
    
    for le_val, total_count in sorted_buckets:
        current_interval_count = total_count - prev_count
        if current_interval_count < 0: current_interval_count = 0 
        
        if le_val == float('inf'):
            label = f"> {prev_le}s"
        else:
            label = f"{prev_le}-{le_val}s"
            
        intervals.append(label)
        counts.append(current_interval_count)
        
        prev_count = total_count
        prev_le = le_val
        
    return intervals, counts

# ==========================================
# MAIN EXECUTION
# ==========================================

# 1. Aggregate Data
aggregated_results = aggregate_logs(LOG_FOLDER, FILE_EXTENSION, METRICS_CONFIG)

# 2. Setup Plotting
num_metrics = len(METRICS_CONFIG)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 6 * num_metrics)) # Increased height slightly
plt.subplots_adjust(hspace=0.5)

if num_metrics == 1:
    axes = [axes]

for idx, (metric_name, title) in enumerate(METRICS_CONFIG):
    ax = axes[idx]
    
    buckets = aggregated_results.get(metric_name, {})
    intervals, counts = buckets_to_intervals(buckets)
    
    if not intervals:
        ax.text(0.5, 0.5, f"No data found for {metric_name}", ha='center', va='center')
        ax.set_title(title)
        continue

    df = pd.DataFrame({'Interval': intervals, 'Count': counts})
    
    # Sort numerically
    def extract_upper(x):
        if ">" in x: return float('inf')
        try: return float(x.split('-')[1].replace('s',''))
        except: return 0

    df['sort'] = df['Interval'].apply(extract_upper)
    df = df.sort_values('sort').drop('sort', axis=1)
    
    # OPTIONAL: If you have too many buckets, uncomment the line below to trim
    # df = df.head(25) 

    # Plot Bar Chart
    index = np.arange(len(df))
    bars = ax.bar(index, df['Count'], color='skyblue', edgecolor='black')
    
    # --- LOG SCALE CONFIGURATION ---
    ax.set_yscale('log')
    
    ax.set_title(f"{title} (All Nodes)", fontsize=14, fontweight='bold')
    ax.set_ylabel('Count (Log Scale)', fontsize=12)
    ax.set_xticks(index)
    ax.set_xticklabels(df['Interval'], rotation=45, ha='right')
    ax.grid(axis='y', linestyle='--', alpha=0.5, which='both') # Grid for log scale

    # Annotate bars
    for i, rect in enumerate(bars):
        height = rect.get_height()
        if height > 0:
            # Adjust label position slightly for log scale visibility
            ax.text(rect.get_x() + rect.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# CONFIGURATION - Transaction Pool Lifecycle
# ==========================================
LOG_FOLDER = "."  
FILE_EXTENSION = ".log" 

# Transaction Pool Lifecycle Metrics (actual transaction flow through the system)
TXPOOL_METRICS_CONFIG = [
    ("substrate_sub_txpool_timing_event_ready", "Time to Ready State"),
    ("substrate_sub_txpool_timing_event_in_block", "Time to Block Inclusion"),
    ("substrate_sub_txpool_timing_event_finalized", "Time to Finalization")
]

# ==========================================
# PARSER LOGIC (Reusing from previous cell)
# ==========================================
def parse_prometheus_buckets(file_content, metric_name):
    """
    Extracts bucket counts for a specific metric from raw Prometheus text.
    Returns a dict: {le_value: count}
    """
    pattern = re.compile(rf'{metric_name}_bucket\{{.*le="([\d\.\+Inf]+)".*\}} (\d+)')
    
    bucket_map = {}
    
    for line in file_content.splitlines():
        if metric_name + "_bucket" not in line:
            continue
            
        match = pattern.search(line)
        if match:
            le_str = match.group(1)
            if "+Inf" in le_str:
                le_val = float('inf')
            else:
                try:
                    le_val = float(le_str)
                except ValueError:
                    continue
            
            count = int(float(match.group(2)))
            
            if le_val in bucket_map:
                bucket_map[le_val] = max(bucket_map[le_val], count)
            else:
                bucket_map[le_val] = count
                
    return bucket_map

def aggregate_logs(folder, extension, metrics_list):
    """
    Scans folder for files and aggregates buckets for all requested metrics.
    Returns a dict: {metric_name: {le_value: total_count}}
    """
    aggregated_data = {m_name: {} for m_name, _ in metrics_list}
    files_found = 0
    
    print(f"Scanning '{folder}' for *{extension} files...")
    
    for filename in os.listdir(folder):
        if filename.endswith(extension) and "prometheus" in filename:
            files_found += 1
            path = os.path.join(folder, filename)
            
            try:
                with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                
                for metric_name, _ in metrics_list:
                    file_buckets = parse_prometheus_buckets(content, metric_name)
                    for le, count in file_buckets.items():
                        current_total = aggregated_data[metric_name].get(le, 0)
                        aggregated_data[metric_name][le] = current_total + count
                        
            except Exception as e:
                print(f"Error reading {filename}: {e}")
                
    print(f"Aggregated metrics from {files_found} log files.")
    return aggregated_data

def buckets_to_intervals(bucket_map):
    """
    Converts cumulative buckets (le) into non-cumulative interval counts.
    """
    if not bucket_map:
        return [], []
    
    sorted_buckets = sorted(bucket_map.items(), key=lambda x: x[0])
    
    intervals = []
    counts = []
    prev_count = 0
    prev_le = 0.0
    
    for le_val, total_count in sorted_buckets:
        current_interval_count = total_count - prev_count
        if current_interval_count < 0: current_interval_count = 0 
        
        if le_val == float('inf'):
            label = f"> {prev_le}s"
        else:
            label = f"{prev_le}-{le_val}s"
            
        intervals.append(label)
        counts.append(current_interval_count)
        
        prev_count = total_count
        prev_le = le_val
        
    return intervals, counts

# ==========================================
# MAIN EXECUTION - Transaction Pool Lifecycle
# ==========================================

print("=" * 60)
print("TRANSACTION POOL LIFECYCLE ANALYSIS")
print("=" * 60)
print("These metrics show transaction timing from submission")
print("through the transaction pool to finalization.\n")

# 1. Aggregate Data
aggregated_results = aggregate_logs(LOG_FOLDER, FILE_EXTENSION, TXPOOL_METRICS_CONFIG)

# 2. Setup Plotting
num_metrics = len(TXPOOL_METRICS_CONFIG)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 6 * num_metrics))
plt.subplots_adjust(hspace=0.5)

if num_metrics == 1:
    axes = [axes]

for idx, (metric_name, title) in enumerate(TXPOOL_METRICS_CONFIG):
    ax = axes[idx]
    
    buckets = aggregated_results.get(metric_name, {})
    intervals, counts = buckets_to_intervals(buckets)
    
    if not intervals:
        ax.text(0.5, 0.5, f"No data found for {metric_name}", ha='center', va='center')
        ax.set_title(title)
        continue

    df = pd.DataFrame({'Interval': intervals, 'Count': counts})
    
    # Sort numerically
    def extract_upper(x):
        if ">" in x: return float('inf')
        try: return float(x.split('-')[1].replace('s',''))
        except: return 0

    df['sort'] = df['Interval'].apply(extract_upper)
    df = df.sort_values('sort').drop('sort', axis=1)
    
    # Plot Bar Chart
    index = np.arange(len(df))
    bars = ax.bar(index, df['Count'], color='lightgreen', edgecolor='black')
    
    # Log Scale
    ax.set_yscale('log')
    
    ax.set_title(f"{title} (All Nodes)", fontsize=14, fontweight='bold')
    ax.set_ylabel('Count (Log Scale)', fontsize=12)
    ax.set_xticks(index)
    ax.set_xticklabels(df['Interval'], rotation=45, ha='right')
    ax.grid(axis='y', linestyle='--', alpha=0.5, which='both')

    # Annotate bars
    for i, rect in enumerate(bars):
        height = rect.get_height()
        if height > 0:
            ax.text(rect.get_x() + rect.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Transaction Pool Lifecycle analysis complete!")

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# CONFIGURATION - Cache Performance Analysis
# ==========================================
LOG_FOLDER = "."  
FILE_EXTENSION = ".log" 

# ==========================================
# PARSER LOGIC FOR COUNTER METRICS
# ==========================================
def parse_prometheus_counter(file_content, metric_name, label_filter=None):
    """
    Extracts counter values for a specific metric from raw Prometheus text.
    Returns a dict: {label_values: count}
    """
    if label_filter:
        pattern = re.compile(rf'{metric_name}\{{.*{label_filter}.*\}} (\d+\.?\d*)')
    else:
        pattern = re.compile(rf'{metric_name}\{{.*\}} (\d+\.?\d*)')
    
    values = {}
    
    for line in file_content.splitlines():
        if metric_name not in line:
            continue
            
        match = pattern.search(line)
        if match:
            value = float(match.group(1))
            
            # Extract labels
            labels_match = re.search(r'\{(.*?)\}', line)
            if labels_match:
                labels = labels_match.group(1)
                values[labels] = value
                
    return values

def aggregate_counter_logs(folder, extension, metric_name, label_filter=None):
    """
    Scans folder for files and aggregates counter values.
    Returns aggregated totals.
    """
    aggregated_data = {}
    files_found = 0
    
    print(f"Scanning '{folder}' for *{extension} files...")
    
    for filename in os.listdir(folder):
        if filename.endswith(extension) and "prometheus" in filename:
            files_found += 1
            path = os.path.join(folder, filename)
            
            try:
                with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                
                file_values = parse_prometheus_counter(content, metric_name, label_filter)
                for labels, value in file_values.items():
                    aggregated_data[labels] = aggregated_data.get(labels, 0) + value
                        
            except Exception as e:
                print(f"Error reading {filename}: {e}")
                
    print(f"Aggregated from {files_found} log files.")
    return aggregated_data

# ==========================================
# MAIN EXECUTION - Cache Performance
# ==========================================

print("=" * 60)
print("CACHE PERFORMANCE ANALYSIS")
print("=" * 60)
print()

# 1. Transaction Validation Cache
print("1. Transaction Validation Cache")
print("-" * 40)

tx_cache_hits = aggregate_counter_logs(LOG_FOLDER, FILE_EXTENSION, "ledger_tx_validation_cache_hits_total")
tx_cache_misses = aggregate_counter_logs(LOG_FOLDER, FILE_EXTENSION, "ledger_tx_validation_cache_misses_total")

# Calculate totals and hit rates
total_hits_soft = sum(v for k, v in tx_cache_hits.items() if 'soft' in k)
total_hits_strict = sum(v for k, v in tx_cache_hits.items() if 'strict' in k)
total_hits = total_hits_soft + total_hits_strict
total_misses = sum(tx_cache_misses.values())
total_attempts = total_hits + total_misses

if total_attempts > 0:
    hit_rate = (total_hits / total_attempts) * 100
    print(f"Total Cache Hits: {int(total_hits):,} (Soft: {int(total_hits_soft):,}, Strict: {int(total_hits_strict):,})")
    print(f"Total Cache Misses: {int(total_misses):,}")
    print(f"Overall Hit Rate: {hit_rate:.2f}%")
    print()

# 2. Trie Cache Performance
print("2. Trie Cache Performance")
print("-" * 40)

# Local cache
local_attempts = aggregate_counter_logs(LOG_FOLDER, FILE_EXTENSION, "trie_cache_local_fetch_attempts")
local_hits = aggregate_counter_logs(LOG_FOLDER, FILE_EXTENSION, "trie_cache_local_hits")

# Shared cache
shared_attempts = aggregate_counter_logs(LOG_FOLDER, FILE_EXTENSION, "trie_cache_shared_fetch_attempts")
shared_hits = aggregate_counter_logs(LOG_FOLDER, FILE_EXTENSION, "trie_cache_shared_hits")

# Calculate hit rates for node and value caches
local_node_attempts = sum(v for k, v in local_attempts.items() if 'node' in k)
local_node_hits = sum(v for k, v in local_hits.items() if 'node' in k)
local_value_attempts = sum(v for k, v in local_attempts.items() if 'value' in k)
local_value_hits = sum(v for k, v in local_hits.items() if 'value' in k)

shared_node_attempts = sum(v for k, v in shared_attempts.items() if 'node' in k)
shared_node_hits = sum(v for k, v in shared_hits.items() if 'node' in k)
shared_value_attempts = sum(v for k, v in shared_attempts.items() if 'value' in k)
shared_value_hits = sum(v for k, v in shared_hits.items() if 'value' in k)

print("\nLocal Cache:")
if local_node_attempts > 0:
    local_node_rate = (local_node_hits / local_node_attempts) * 100
    print(f"  Node Cache: {int(local_node_hits):,}/{int(local_node_attempts):,} = {local_node_rate:.2f}% hit rate")
if local_value_attempts > 0:
    local_value_rate = (local_value_hits / local_value_attempts) * 100
    print(f"  Value Cache: {int(local_value_hits):,}/{int(local_value_attempts):,} = {local_value_rate:.2f}% hit rate")

print("\nShared Cache:")
if shared_node_attempts > 0:
    shared_node_rate = (shared_node_hits / shared_node_attempts) * 100
    print(f"  Node Cache: {int(shared_node_hits):,}/{int(shared_node_attempts):,} = {shared_node_rate:.2f}% hit rate")
if shared_value_attempts > 0:
    shared_value_rate = (shared_value_hits / shared_value_attempts) * 100
    print(f"  Value Cache: {int(shared_value_hits):,}/{int(shared_value_attempts):,} = {shared_value_rate:.2f}% hit rate")

# 3. Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Transaction Validation Cache
if total_attempts > 0:
    ax1 = axes[0]
    labels = ['Hits (Soft)', 'Hits (Strict)', 'Misses']
    sizes = [total_hits_soft, total_hits_strict, total_misses]
    colors = ['lightgreen', 'green', 'lightcoral']
    explode = (0.05, 0.05, 0.1)
    
    ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', explode=explode, startangle=90)
    ax1.set_title('Transaction Validation Cache\nHit/Miss Distribution', fontsize=14, fontweight='bold')

# Plot 2: Local Trie Cache Hit Rates
ax2 = axes[1]
cache_types = ['Local\nNode', 'Local\nValue']
hit_rates = []
if local_node_attempts > 0:
    hit_rates.append((local_node_hits / local_node_attempts) * 100)
else:
    hit_rates.append(0)
if local_value_attempts > 0:
    hit_rates.append((local_value_hits / local_value_attempts) * 100)
else:
    hit_rates.append(0)

bars = ax2.bar(cache_types, hit_rates, color=['steelblue', 'skyblue'], edgecolor='black')
ax2.set_ylabel('Hit Rate (%)', fontsize=12)
ax2.set_title('Local Trie Cache Hit Rates', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 100)
ax2.grid(axis='y', linestyle='--', alpha=0.5)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Plot 3: Shared Trie Cache Hit Rates
ax3 = axes[2]
cache_types = ['Shared\nNode', 'Shared\nValue']
hit_rates = []
if shared_node_attempts > 0:
    hit_rates.append((shared_node_hits / shared_node_attempts) * 100)
else:
    hit_rates.append(0)
if shared_value_attempts > 0:
    hit_rates.append((shared_value_hits / shared_value_attempts) * 100)
else:
    hit_rates.append(0)

bars = ax3.bar(cache_types, hit_rates, color=['darkgreen', 'lightgreen'], edgecolor='black')
ax3.set_ylabel('Hit Rate (%)', fontsize=12)
ax3.set_title('Shared Trie Cache Hit Rates', fontsize=14, fontweight='bold')
ax3.set_ylim(0, 100)
ax3.grid(axis='y', linestyle='--', alpha=0.5)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("✅ Cache performance analysis complete!")
print("=" * 60)